In [3]:
import fitz  # PyMuPDF
import pandas as pd
import json
import chardet
from time import sleep
import time
import requests
import pandas as pd
import random
import pymysql
import json
import sys
import sqlalchemy
from datetime import datetime,date, timedelta
import os
import openai
from pathlib import Path
from sqlalchemy import text
from datetime import datetime
import shutil
import os
import shutil
import re
import json
from urllib.parse import urlparse, parse_qs
from urllib import parse

def parse_jcurl(url):
    # 解析URL中的查询参数
    parsed_url = urlparse(url)
    query_params = parse_qs(parsed_url.query)
    announceId=int(query_params['announcementId'][0])
    announceTime=query_params['announcementTime'][0]

    url0=f'http://www.cninfo.com.cn/new/announcement/bulletin_detail?announceId={announceId}&flag=true&announceTime={announceTime}'
    hd = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6',
        'Connection': 'keep-alive',
        'Content-Length': '0',
        # 'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
        'Cookie': 'JSESSIONID=32094F939AC5788AE54B85584B1E247E; insert_cookie=45380249; routeId=.uc1; SID=2ba93430-8d4d-47c0-9db5-bceb6cb7e8ba; _sp_ses.2141=*; SF_cookie_4=90357558; _sp_id.2141=6c21141c-7ece-48ea-9323-ac846752aaca.1697523969.13.1720139400.1719994574.b8bf94a9-7bc1-458d-9183-fd36bb5dc6d6',
        'Host': 'www.cninfo.com.cn',
        'Origin': 'http://www.cninfo.com.cn',
        #'Pragma': 'no-cache',
        'Referer': url0,
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 Edg/126.0.0.0',
        }
    data={
      'announceId': announceId,
      'flag': 'true',
      'announceTime': announceTime,
    }
    r = requests.post(url0,headers=hd,data=data)
    r = str(r.content, encoding="utf-8")
    r = json.loads(r)
    fileurl=r['announcement']['announcementTitle']
    filename=r['fileUrl']
    return fileurl,filename

sql_engine = sqlalchemy.create_engine(
'mysql+pymysql://%s:%s@%s:%s/%s' % (
    'hz_work',
    'Hzinsights2015',
    'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
    '3306',
    'yq',
), poolclass=sqlalchemy.pool.NullPool
)
sql_engine1 = sqlalchemy.create_engine(
'mysql+pymysql://%s:%s@%s:%s/%s' % (
    'root',
    's5bx55dh',
    'bja.sealos.run',
    '44525',
    'timinfo',
), poolclass=sqlalchemy.pool.NullPool
)

destination_folder = r'D:\2024年\企业预警通'
destination_folder1 = r'D:\2024年\小文件'

def download_PDF(fileUrl,fileName):  #下载pdf
  file_path = os.path.join(destination_folder,fileName+".pdf")
  # 检查文件是否已存在，如果存在则不下载
  if os.path.exists(file_path):
    print(f"{fileName}File already exists!")
  else:
    r = requests.get(fileUrl)
    with open(file_path, "wb") as f:
      f.write(r.content)
    print(f"Downloaded: {fileName}.pdf\n")

def update_database2(originfilename,fileurl,targetfilename):
  sql = """
  UPDATE 23年财报文件
  set fileName= :targetfilename,
  fileUrl= :fileurl
  where fileName = :originfilename
  """
  # 构建参数字典
  params = {
    "originfilename":originfilename,
    "fileurl":fileurl,
    "targetfilename":targetfilename
  }
  with sql_engine.begin() as connection:
    connection.execute(text(sql), params)
def beizhu_database(originfilename,beizhu):
  sql = """
  UPDATE 23年财报文件
  set 备注= :beizhu
  where fileName = :originfilename
  """
  # 构建参数字典
  params = {
    "originfilename":originfilename,
    "beizhu":beizhu
  }
  with sql_engine.begin() as connection:
    connection.execute(text(sql), params)


def extract_links(pdf_path):
    doc = fitz.open(pdf_path)
    links = []

    for page in doc:
        annots = page.annots()
        for annot in annots:
            if annot.type[0] == 1:  # 类型为链接
                url = annot.uri  # 提取链接URL
                links.append(url)
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text = page.get_text("text")  # 获取页面文本
        
        # 使用正则表达式匹配URL
        url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
        urls = url_pattern.findall(text)
        
        links.extend(urls)  # 将找到的URL添加到列表中
    return links
sql="""
SELECT fileName
from yq.23年财报文件
where fileName !=''"""

with sql_engine.begin() as connection:
    df = pd.read_sql(sql, connection)
if not os.path.exists(destination_folder1):
    os.makedirs(destination_folder1)

pattern1 = r"审计报告|年度报告|年报|报表"
pattern2 = r"专项审计|摘要|page|风险提示|提示性|补充公告|更正|鉴证报告|代理事务|增信|担保|监管|说明"
one_mb = 2 * 1024 * 1024  # 1 MB in bytes
for filename in os.listdir(destination_folder):
    if filename in df['fileName']:
       continue
    print(filename)

    filepath=os.path.join(destination_folder,filename)
    file_size = os.path.getsize(filepath)
    
    if file_size > one_mb:
        continue
    shutil.copy(filepath,destination_folder1)
    beizhu_database(filename,'小于2mb')
    # is_link=0
    # try:
    #   links=extract_links(filepath)
    #   is_link=1
    #   beizhu_database(filename,'提取超链接成功')
    # except:
    #   beizhu_database(filename,'提取超链接失败')
    # if is_link==1 and len(links)>0:
    #   for link in links:
    #       print(link)
    #       try:
    #         fileurl,targetfilename=parse_jcurl(link)
    #       except:
    #         print('该超链接无效')
    #         continue
    #       if re.search(pattern1,targetfilename) and not (re.search(pattern2,targetfilename)):
    #         download_PDF(fileurl,targetfilename)
    #         update_database2(filename,fileurl,targetfilename)
    #         os.remove(filepath)
    #         break
      
    

07_AFS_中冶交通建设集团有限公司_合并审计报告_20231231.pdf
1-孝昌县顺和开发投资有限责任公司公司债券年度报告（2023年）.pdf
16万通02-关于广西万通房地产有限公司发生重大损失、被出具无法表示意见及间接控股股东刊发2023年度报告的临时受托管理事务报告.pdf
17广业01：广东省环保集团有限公司公司债券2023年度审计报告.pdf
19华强05-深圳华强集团有限公司2023年度审计报告.pdf
19道桥01-遵义道桥建设（集团）有限公司公司债券年度报告（2023年）.pdf
2019年长沙金霞经济开发区开发建设总公司企业债券2023年度担保人财务报告及附注.pdf
2023年三亚环境投资集团有限公司审计报告.pdf
2023年度崇仁县城市建设投资发展有限公司公司债券2023年年度财务报表及附注.pdf
2023年度建信金融租赁有限公司已审财务报表.pdf
2023年度泾县国有资产投资运营有限公司审计报告.pdf
2023年担保人审计报告（湖北省融资担保集团有限责任公司）.pdf
2023年远洋资本有限公司审计报告.pdf
2024-04-08-143734.SH-18金隅02-北京金隅集团股份有限公司2023年度审计报告.pdf
2024-04-13-600518.SH-ST康美-600518康美药业2023年年度报告.pdf
2024-04-23-102383165.IB-23中铝国工MTN002-中铝国际工程股份有限公司2023年年度报告.pdf
2024-04-30-152077.SH-PR安方债-毕节市安方建设投资(集团)有限公司2023年度审计报告.pdf
20DFCX01-天津东方财信投资集团有限公司公司债券2023年度报告.pdf
20兴兰01-兰考县兴兰农村投资发展有限公司公司债券2023年度报告.pdf
20兴宁01-天津市宁河区兴宁建设投资集团有限公司公司债券2023年度报告.pdf
20天乾01-湖北天乾资产管理有限公司公司债券2023年度报告.pdf
20奉开01-宁波市雪窦开发投资集团有限公司公司债券2023年度报告.pdf
20广开04-广州开发区控股集团有限公司2023年审计报告.pdf
20徐国盛-徐州市国盛控股集团有限公司公司债券2023年度报告、财务报告及担保人财报.pdf
20新工01-新昌县工业

In [2]:
import requests
url='http://www.cninfo.com.cn/new/disclosure/detail?plate=szse&orgId=9900014189&stockCode=002466&announcementId=1219430722&announcementTime=2024-03-28'
data = {
announceId: 1219430722
flag: true
announceTime: 2024-03-28,
}
r = requests.get(url)

In [33]:
import json
from urllib.parse import urlparse, parse_qs
from urllib import parse
url = 'http://www.cninfo.com.cn/new/disclosure/detail?plate=szse&orgId=9900014189&stockCode=002466&announcementId=1219430722&announcementTime=2024-03-28'


def parse_jcurl(url):
    # 解析URL中的查询参数
    parsed_url = urlparse(url)
    query_params = parse_qs(parsed_url.query)
    announceId=int(query_params['announcementId'][0])
    announceTime=query_params['announcementTime'][0]

    url0=f'http://www.cninfo.com.cn/new/announcement/bulletin_detail?announceId={announceId}&flag=true&announceTime={announceTime}'
    hd = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6',
        'Connection': 'keep-alive',
        'Content-Length': '0',
        # 'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
        'Cookie': 'JSESSIONID=32094F939AC5788AE54B85584B1E247E; insert_cookie=45380249; routeId=.uc1; SID=2ba93430-8d4d-47c0-9db5-bceb6cb7e8ba; _sp_ses.2141=*; SF_cookie_4=90357558; _sp_id.2141=6c21141c-7ece-48ea-9323-ac846752aaca.1697523969.13.1720139400.1719994574.b8bf94a9-7bc1-458d-9183-fd36bb5dc6d6',
        'Host': 'www.cninfo.com.cn',
        'Origin': 'http://www.cninfo.com.cn',
        #'Pragma': 'no-cache',
        'Referer': url0,
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 Edg/126.0.0.0',
        }
    r = requests.post(url0,headers=hd,data=data)
    r = str(r.content, encoding="utf-8")
    r = json.loads(r)
    fileurl=r['announcement']['announcementTitle']
    filename=r['fileUrl']
    return fileurl,filename
parse_jcurl(url)

('2023年年度审计报告',
 'http://static.cninfo.com.cn/finalpage/2024-03-28/1219430722.PDF')

{'announcement': {'id': None,
  'secCode': '002466',
  'secName': '天齐锂业',
  'orgId': '9900014189',
  'announcementId': '1219430722',
  'announcementTitle': '2023年年度审计报告',
  'announcementTime': 1711555200000,
  'adjunctUrl': 'finalpage/2024-03-28/1219430722.PDF',
  'adjunctSize': 19853,
  'adjunctType': 'PDF',
  'storageTime': 1711552679000,
  'columnId': '09020202||250101||251302',
  'pageColumn': None,
  'announcementType': '01010903||010112||012913',
  'associateAnnouncement': None,
  'important': False,
  'batchNum': None,
  'announcementContent': None,
  'orgName': None,
  'tileSecName': None,
  'shortTitle': None,
  'announcementTypeName': None,
  'secNameList': None},
 'fileUrl': 'http://static.cninfo.com.cn/finalpage/2024-03-28/1219430722.PDF',
 'bulletinStatus': None,
 'columIdentify': 'qw'}

In [7]:
from fuzzywuzzy import fuzz
import json
import os
from time import sleep
from urllib import parse
import requests
import urllib.request
import string
import re
import pandas as pd
from datetime import date, datetime,timedelta
import urllib.parse
from sqlalchemy import text
import sqlalchemy
import random
import shutil
import sys
import os

# 获取模块的绝对路径
module_path = r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\弘则"

# 检查模块路径是否已经在 sys.path 中，避免重复添加
if module_path not in sys.path:
    # 将模块路径添加到 sys.path 开头，确保优先加载这个路径的模块
    sys.path.insert(0, module_path)

import timai
from timai import get_html_text_and_images
from timai import client_hz
from timai import summarize_content,pro_db
df=pro_db(type='df',sql='select list_keywords from yq.金融债舆情监控')
# list_keywors=df['list_keywords'].tolist()
flat_list = [
    keyword.strip() 
    for keywords in df['list_keywords']
    for keyword in keywords.split(',')
    if keyword.strip()
]
target_company = "中国民生信托"

# 模糊匹配函数，可以设定一个匹配阈值，比如75%
def count_fuzzy_matches(target, companies, threshold=75):
    count = 0
    for company in companies:
        # fuzz.token_set_ratio更适用于处理有无序、元素重复的情况，这里作为示例
        similarity = fuzz.token_set_ratio(target, company)
        if similarity >= threshold:
            count += 1
    return count

# 使用函数计算目标公司在列表中出现的次数
matched_count = count_fuzzy_matches(target_company, flat_list)

print(f"'{target_company}' 出现了 {matched_count} 次。")

'中国民生信托' 出现了 1 次。
